# CAREPILOT — LLM Operational Explanation Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

Task 9 — local LLM layer that converts aggregated AI signals (Tasks 1–8) into concise enterprise ITSM explanations.

## 1. Objective

Generate **concise operational explanations** from aggregated CAREPILOT AI predictions using a **free local LLM** (Ollama). Agents receive a single decision-support narrative covering escalation, transfer, action, SLA, priority, KM, and missing-information signals — without replacing deterministic ML outputs.

**Flow:** Ticket → AI predictions → Local LLM explanation → Operational decision support

## 2. Problem Type

- **LLM qualitative evaluation** (not classification or retrieval)
- **Input:** structured ticket + AI signal context block
- **Output:** numbered sections — operational summary, recommended action, reasoning, missing information, next steps
- **Evaluation:** rubric-based scoring (1–5) + latency + hallucination flag

## 3. Dataset Description

| Property | Value |
| --- | --- |
| Evaluation cases | 50 representative scenarios |
| Scenario types | escalation, transfer, SLA, KM, missing info, local resolution |
| Input | Ticket fields + Tasks 1–8 AI outputs |
| Models evaluated | Mistral, Llama3, DeepSeek R1 (Ollama local) |
| Inference | `POST /api/generate` · temperature 0.15 · max ~420 tokens |

Cases are designed to mirror real Carrefour store ITSM situations: checkout failures, network issues, EMV declines, vague tickets, and local resolutions.

## 4. Preprocessing & Prompt Design

1. **Context assembly** — ticket title, description, store, perimeter, priority + all AI prediction strings
2. **Enterprise system prompt** — use ONLY provided context; no invented facts; no first person; no model internals
3. **Output format** — five numbered sections enforced by prompt template
4. **Generation params** — temperature 0.15, `num_predict` ~420 for concise responses
5. **Parsing** — section extraction + automated rubric scoring per dimension

When Ollama is offline, a deterministic template fallback preserves functionality without hallucination risk.

## 5. Models Compared

1. **Mistral** (`mistral:latest`, 4.4 GB) — default production model
2. **Llama3** (`llama3:latest`, 4.7 GB) — alternative open-weight instruct model
3. **DeepSeek R1** (`deepseek-r1:latest`, 5.2 GB) — reasoning-oriented model; evaluated if hardware permits

All models run locally via Ollama — no paid API dependency.

## 6. Evaluation Metrics

| Dimension | Description |
| --- | --- |
| **Clarity** | Readable, structured, professional ITSM language |
| **Operational Relevance** | References provided signals; avoids generic filler |
| **Completeness** | All five sections present and substantive |
| **Hallucination Control** | No invented stores, teams, codes, or procedures |
| **Avg Latency (ms)** | End-to-end Ollama generation time |
| **Overall Score** | Weighted rubric aggregate (1–5, capped at 4.8) |

**Hallucination rate** = fraction of responses flagged for invented entities not in context.

## 7. Inference Pipeline

```
Ticket + AI predictions (Tasks 1–8)
        → buildExplanationContextBlock()
        → enterprise system prompt + user prompt
        → Ollama POST /api/generate
        → parseExplanationResponse()
        → store in cp_agent.ai_operational_explanations
```

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break
else:
    raise FileNotFoundError("Project root not found")

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, display_plots, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
BENCHMARKS = ROOT / "ml" / "benchmarks"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

RUN_BENCHMARK = False


In [ ]:
script = BENCHMARKS / "llm_explanation" / "benchmark.py"
if RUN_BENCHMARK:
    subprocess.run([sys.executable, str(script), "--project-root", str(ROOT)], check=True)
else:
    print("Loading saved LLM benchmark artifacts (requires prior Ollama benchmark run).")
    print(f"  python {script} --project-root {ROOT}")


## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "03_llm_explanation_benchmark_metrics.json"
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
rows = payload["results"]

results_df = pd.DataFrame(rows)
label_map = {
    "model": "Model",
    "clarity": "Clarity",
    "relevance": "Operational Relevance",
    "completeness": "Completeness",
    "hallucination_control": "Hallucination Control",
    "avg_latency_ms": "Avg Latency (ms)",
    "overall": "Overall Score",
    "hallucination_rate": "Hallucination Rate",
}
results_df = results_df.rename(columns=label_map)
display_cols = [c for c in label_map.values() if c in results_df.columns]
display(Markdown(f"### Qualitative benchmark ({payload['cases_evaluated']} cases · Ollama live: {payload['ollama_available']})"))
display(results_df[display_cols])


## 9. Benchmark Results — Report Table

In [ ]:
def display_name(model: str) -> str:
    key = model.split(":")[0].lower()
    return {"mistral": "Mistral", "llama3": "Llama3", "deepseek-r1": "DeepSeek R1"}.get(key, model)

figure_df = pd.DataFrame(
    [
        {
            "Model": display_name(r["model"]),
            "Clarity": f"{r['clarity']:.2f}",
            "Operational Relevance": f"{r['relevance']:.2f}",
            "Completeness": f"{r['completeness']:.2f}",
            "Hallucination Control": f"{r['hallucination_control']:.2f}",
            "Avg Latency": f"{r['avg_latency_ms']:.0f} ms",
            "Overall Score": f"{r['overall']:.2f}",
        }
        for r in rows
    ]
)

png_path = PLOTS / "03_llm_explanation_benchmark_results_table.png"
plot_results_table(
    figure_df,
    title="CarePilot LLM Operational Explanation Benchmark",
    footer="50 evaluation tickets · Live Ollama inference · Qualitative rubric (1–5)",
    output_path=png_path,
)
display_plot(png_path)


## 10. Visualizations

In [ ]:
chart_df = pd.DataFrame(rows)
chart_df["label"] = chart_df["model"].map(lambda m: display_name(m))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
metrics = ["clarity", "relevance", "completeness", "hallucination_control"]
x = chart_df["label"]
width = 0.18
for i, metric in enumerate(metrics):
    axes[0].bar([j + i * width for j in range(len(x))], chart_df[metric], width=width, label=metric.replace("_", " ").title())
axes[0].set_xticks([j + 1.5 * width for j in range(len(x))])
axes[0].set_xticklabels(x)
axes[0].set_ylim(0, 5)
axes[0].set_title("Quality Dimensions by Model")
axes[0].legend(fontsize=8, loc="lower right")
axes[0].set_ylabel("Rubric score (1–5)")

colors = ["#173b6c", "#46698d", "#6385a9"][: len(chart_df)]
axes[1].bar(chart_df["label"], chart_df["avg_latency_ms"], color=colors)
axes[1].set_title("Average Latency by Model")
axes[1].set_ylabel("Milliseconds")
for i, v in enumerate(chart_df["avg_latency_ms"]):
    axes[1].text(i, v + 200, f"{v:.0f} ms", ha="center", fontsize=9)

fig.suptitle("LLM Explanation Benchmark — Quality vs Latency", y=1.02, fontweight="bold", color="#173b6c")
fig.tight_layout()
plt.show()


## 11. Example Evaluation Case

In [ ]:
cases = json.loads((BENCHMARKS / "llm_explanation" / "eval_cases.json").read_text(encoding="utf-8"))
example = cases[0]
display(Markdown(f"**Case {example['case_id']}** - scenario: `{example['scenario_type']}`"))
display(Markdown("```\n" + example["context_block"] + "\n```"))


### Expected explanation structure (Mistral production output)

1. **Operational summary** - ticket situation at named store, perimeter context
2. **Recommended action** - derived from action/escalation/transfer AI signals
3. **Reasoning** - ties ML predictions together without inventing facts
4. **Missing information** - from Task 8 completeness analysis
5. **Next steps** - concrete agent actions (validate, request info, escalate, transfer)

**Example production excerpt (live Mistral inference):**

> *Operational summary:* Local orchestration service at Carrefour Market Pau stopped after patch reboot; dependency chain monitored.
>
> *Recommended action:* Transfer to Store Infrastructure team.
>
> *Reasoning:* Transfer prediction required (100% confidence); resolution confidence locally low (16%).

## 12. Interpretation

| Model | Overall | Latency | Assessment |
| --- | --- | --- | --- |
| **Mistral** | 4.07 | 7,847 ms | Best latency/quality trade-off |
| Llama3 | 4.07 | 9,622 ms | Tied quality; slower |
| DeepSeek R1 | 3.86 | 12,519 ms | Verbose reasoning; lower rubric scores |

- **Mistral and Llama3** tie on overall score (~4.07) with strong completeness (~4.55) and zero hallucination flags
- **Mistral is ~18% faster** than Llama3 with the smallest model footprint (4.4 GB)
- **DeepSeek R1** scores lower on clarity and operational relevance — reasoning traces add latency without quality gain for this structured prompt
- **Hallucination control** stable at 4.0 / 0% flagged across all models thanks to strict context-only prompting

Warm inference after model load: typically 4–10 s per explanation; acceptable for batch generation and on-demand ticket detail views.

## 13. Selected Model

**Selected for CAREPILOT: Mistral** (`OLLAMA_EXPLANATION_MODEL=mistral`)

```env
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_EXPLANATION_MODEL=mistral
```

- 70 production tickets generated with live Mistral inference (no template fallback)
- Stored in `cp_agent.ai_operational_explanations`
- UI: minimal operational panel (not chatbot interface)

## 14. Limitations

- Latency depends on local hardware (CPU/GPU); cold start can exceed 60 s
- Qualitative rubric is automated — human agent review recommended for PFE validation
- Template fallback used when Ollama is offline (deterministic, no invented facts)
- DeepSeek R1 not recommended for production given latency and lower rubric scores
- Explanation quality bounded by upstream AI signal quality (Tasks 1–8)

## 15. Generated Artifacts

- `ml/artifacts/03_llm_explanation_benchmark_metrics.json`
- `ml/benchmarks/llm_explanation/eval_cases.json`
- `ml/plots/03_llm_explanation_benchmark_results_table.png`
- `ml/reports/03_llm_operational_explanation_summary.md`
- Production: `cp_agent.ai_operational_explanations`